In [23]:
import pandas as pd 
import matplotlib.pyplot as plt
import datetime as dt

In [67]:
df = pd.read_csv('data/vendas_loja_informatica.csv')

In [1]:
#Informações sobre o Dataset
df.shape
df.dtypes
df.head(10)

# Remover linhas duplicadas
df = df.drop_duplicates()

df.isnull()

# Corrigir preço (vírgula -> ponto, converter pra número)
df["preco"] = df["preco"].astype(str).str.replace(",", ".", regex=False)
df["preco"] = pd.to_numeric(df["preco"], errors="coerce")

# Preencher cliente e cidade vazios (agora com atribuição, senão não salva)
df["cliente"] = df["cliente"].fillna("não informado")

df["cidade"] = df["cidade"].str.strip().str.title()
df["cidade"] = df["cidade"].fillna("não informado")
df["cidade"] = df["cidade"].replace({"Sp": "São Paulo"})

import re  # módulo de expressões regulares, usado para identificar padrões de texto

# Dicionário para traduzir nomes de meses em português para o número (com 2 dígitos)
meses = {
    'janeiro': '01', 'fevereiro': '02', 'março': '03', 'abril': '04',
    'maio': '05', 'junho': '06', 'julho': '07', 'agosto': '08',
    'setembro': '09', 'outubro': '10', 'novembro': '11', 'dezembro': '12'
}

def normalizar_data(valor):
    # Função aplicada a cada célula da coluna "data", uma por vez

    if not isinstance(valor, str):
        return pd.NaT  # se não for texto (ex: já é NaN/vazio), retorna "data ausente" e encerra

    v = valor.strip()  # remove espaços extras no início/fim do texto

    # --- Tenta reconhecer datas por extenso, ex: "12 de janeiro de 2026" ---
    m = re.match(r'(\d{1,2}) de (\w+) de (\d{4})', v.lower())
    # \d{1,2}  -> captura o dia (1 ou 2 dígitos)
    # \w+      -> captura o nome do mês (letras)
    # \d{4}    -> captura o ano (4 dígitos)
    # .lower() -> deixa tudo minúsculo pra bater com o dicionário "meses"

    if m:  # se o padrão por extenso foi encontrado
        dia, mes_nome, ano = m.groups()  # separa os 3 grupos capturados em variáveis
        mes = meses.get(mes_nome)  # traduz o nome do mês pro número correspondente
        if mes:  # se o mês foi reconhecido no dicionário
            return pd.Timestamp(f"{ano}-{mes}-{dia.zfill(2)}")
            # zfill(2) garante 2 dígitos no dia (ex: "5" -> "05")
            # monta a data no formato ISO e converte pra um Timestamp de verdade

    # --- Tenta reconhecer formato ISO, ex: "2026-01-07" ---
    if re.match(r'^\d{4}-\d{2}-\d{2}$', v):
        # ^ e $ garantem que a string inteira segue esse padrão, do início ao fim
        return pd.to_datetime(v, format="%Y-%m-%d", errors="coerce")
        # format explícito evita que o pandas "adivinhe" errado

    # --- Tenta reconhecer formato brasileiro, ex: "05/01/2026" ---
    if re.match(r'^\d{2}/\d{2}/\d{4}$', v):
        return pd.to_datetime(v, format="%d/%m/%Y", errors="coerce")

    # Se não bateu com nenhum formato conhecido, assume como data inválida/ausente
    return pd.NaT

# Aplica a função normalizar_data em cada valor da coluna "data"
df["data"] = df["data"].apply(normalizar_data)

# Converte as datas (já como objetos datetime) para texto no formato dd/mm/yyyy
df["data"] = df["data"].dt.strftime("%d/%m/%Y")

df[df["preco"].isnull()]

# Preencher preço faltante do Fone de Ouvido com a média do próprio produto
media_fone = df[df["produto"] == "Fone de Ouvido"]["preco"].mean()
df.loc[(df["produto"] == "Fone de Ouvido") & (df["preco"].isnull()), "preco"] = media_fone

media_SSD = df[df["produto"] == "SSD 480GB"]["preco"].mean()
df.loc[(df["produto"] == "SSD 480GB") & (df["preco"].isnull()), "preco"] = media_SSD

df[df["produto"] == "Fone de Ouvido"]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

df

df.to_excel("dados_tratados.xlsx")

NameError: name 'df' is not defined

In [ ]:
df.data

In [74]:
df.head(50)

,id_venda,data,produto,categoria,quantidade,preco,cliente,cidade
0,1,05/01/2026,Mouse Gamer,Periféricos,1,89.900000,Ana Souza,Recife
1,2,06/01/2026,Teclado Mecânico,Periféricos,2,259.900000,Bruno Lima,Recife
2,3,07/01/2026,Monitor 24pol,Hardware,3,819.000000,Carla Dias,Recife
3,4,08/01/2026,SSD 480GB,Hardware,4,229.000000,Diego Alves,São Bento Do Una
4,5,09/01/2026,Notebook i5,Hardware,1,3309.000000,Elaine Melo,Sao Bento Do Una
5,6,10/01/2026,Fone de Ouvido,Acessórios,2,149.900000,Fábio Rocha,São Paulo
6,7,11/01/2026,Webcam Full HD,Acessórios,3,159.000000,Gabriela Nunes,São Paulo
7,8,12/01/2026,Cadeira Gamer,Acessórios,4,909.000000,Heitor Costa,São Paulo
8,9,13/01/2026,Placa de Vídeo,Hardware,1,1919.000000,Isabela Prado,Caruaru
9,10,14/01/2026,Memória RAM 8GB,Hardware,2,189.000000,não informado,Caruaru


In [6]:
import os
print(os.path.exists("dados_tratados.xlsx"))
print(os.path.abspath("dados_tratados.xlsx"))
os.startfile("dados_tratados.xlsx")

True
c:\Users\lison\OneDrive\Desktop\aula de vanthuir\dados_tratados.xlsx
